# ROGII - Wellbore Geology Prediction - Stage 8 submission notebook

Self-contained: no internet access needed (Code Competition requirement).

**Stage 8: Stage 4a + a cross-well spatial prior feature**, adapted from a public kernel
(`lightningv08/rogii-dual-pipeline-self-verifying`, see `../context/external-kernels/README.md`).
Every prior stage treated each well in complete isolation; this is the first to use
information from OTHER wells.

**The idea:** wells drilled near each other in (X, Y) tend to hit the same geological
formations at similar depths. For any (X, Y), fit a locally-weighted plane through the K=10
nearest OTHER wells' known formation-contact depths (train-only columns), predict the
formation depth at that location, then derive a candidate TVT from it - a `spatial_tvt_prior`
feature added alongside Stage 4a's existing 11.

**Two attempts before this one:**
1. Raw plane-fit estimate, no guard - RMSE **22,032** ALONE (catastrophically unstable:
   ill-conditioned local plane fits, when the K neighbor wells are near-collinear in (X, Y),
   produce wild extrapolated coefficients; confirmed on well `09ec2ca9`, raw estimate
   -51,563 ft against a true value of ~11,051 ft).
2. **This config** - guarded: fall back to the already-trusted `linear_prior` for any row
   whose spatial estimate disagrees with it by more than 200 ft (mirrors the Stage 7
   guarded-override pattern). Raw feature alone RMSE dropped to **70.09** (sane scale).

**Local validation (5-fold GroupKFold OOF): 52.57** vs Stage 4a's 52.90 baseline - a real
improvement in **every single fold**, not just on average. Genuine, reinforcing signal.

In [ ]:
import glob
import os
import time

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression

RANDOM_STATE = 42
HALF_WINDOW = 5
SEARCH_EXTRA_FT = 100.0
ACCEPT_SLACK = 1.5
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10
SPATIAL_GUARD_FT = 200.0

_KAGGLE_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
DATA_DIR = _KAGGLE_DIR if os.path.isdir(_KAGGLE_DIR) else os.path.join("..", "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
SAMPLE_SUBMISSION = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"DATA_DIR = {DATA_DIR}")
assert os.path.isdir(TRAIN_DIR), f"train dir not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"test dir not found: {TEST_DIR}"

In [ ]:
def list_wells(split_dir):
    files = glob.glob(os.path.join(split_dir, "*__horizontal_well.csv"))
    return sorted(os.path.basename(f).split("__")[0] for f in files)


def linear_prior_predict(known, eval_rows):
    if len(known) < 2:
        fallback = known["TVT_input"].mean() if len(known) else np.nan
        return np.full(len(eval_rows), fallback)
    model = LinearRegression()
    model.fit(known[["MD", "Z"]], known["TVT_input"])
    return model.predict(eval_rows[["MD", "Z"]])


def windowed_shape_match(prior_preds, eval_gr, tw_tvt, tw_gr,
                          half_win=HALF_WINDOW, search_extra=SEARCH_EXTRA_FT,
                          accept_slack=ACCEPT_SLACK):
    n = len(prior_preds)
    if n == 0:
        return prior_preds
    gr_filled = pd.Series(eval_gr).interpolate(limit_direction="both").to_numpy()
    if np.isnan(gr_filled).all():
        return prior_preds
    refined = prior_preds.copy()
    match_err = np.full(n, np.nan)
    for i in range(n):
        lo_row, hi_row = max(0, i - half_win), min(n, i + half_win + 1)
        local_gr = gr_filled[lo_row:hi_row]
        L = len(local_gr)
        if np.isnan(local_gr).any() or L < 3:
            continue
        center_prior = prior_preds[i]
        lo_idx = np.searchsorted(tw_tvt, center_prior - search_extra)
        hi_idx = np.searchsorted(tw_tvt, center_prior + search_extra)
        if hi_idx - lo_idx < L:
            continue
        seg_gr = tw_gr[lo_idx:hi_idx]
        seg_tvt = tw_tvt[lo_idx:hi_idx]
        windows = np.lib.stride_tricks.sliding_window_view(seg_gr, L)
        sse = np.sum((windows - local_gr[None, :]) ** 2, axis=1)
        best = int(np.argmin(sse))
        center_offset = i - lo_row
        refined[i] = seg_tvt[best + center_offset]
        match_err[i] = sse[best] / L
    valid = ~np.isnan(match_err)
    if valid.sum() == 0:
        return prior_preds
    thresh = np.nanmedian(match_err[valid]) * accept_slack
    keep = valid & (match_err <= thresh)
    return np.where(keep, refined, prior_preds)


FEATURE_COLS = [
    "MD", "X", "Y", "Z", "GR", "linear_prior", "windowed_match",
    "match_minus_prior", "dist_from_known_boundary", "eval_zone_frac",
    "known_zone_rows",
]
AUG_FEATURE_COLS = FEATURE_COLS + ["spatial_tvt_prior"]


def build_well_features(well, split_dir, has_target):
    hw = pd.read_csv(os.path.join(split_dir, f"{well}__horizontal_well.csv")).reset_index(drop=True)
    tw_path = os.path.join(split_dir, f"{well}__typewell.csv")
    tw = pd.read_csv(tw_path).dropna(subset=["TVT", "GR"]).sort_values("TVT")

    known = hw[hw["TVT_input"].notna()]
    eval_rows = hw[hw["TVT_input"].isna()]
    if len(eval_rows) == 0:
        return None

    linear_prior = linear_prior_predict(known, eval_rows)

    if len(tw) >= HALF_WINDOW * 2 + 1:
        tw_tvt = tw["TVT"].to_numpy()
        tw_gr = tw["GR"].to_numpy()
        eval_gr = eval_rows["GR"].to_numpy()
        windowed_match = windowed_shape_match(linear_prior, eval_gr, tw_tvt, tw_gr)
    else:
        windowed_match = linear_prior.copy()

    known_md_max = known["MD"].max() if len(known) else eval_rows["MD"].min()
    n_eval = len(eval_rows)

    df = pd.DataFrame({
        "well": well,
        "row_idx": eval_rows.index.to_numpy(),
        "MD": eval_rows["MD"].to_numpy(),
        "X": eval_rows["X"].to_numpy(),
        "Y": eval_rows["Y"].to_numpy(),
        "Z": eval_rows["Z"].to_numpy(),
        "GR": eval_rows["GR"].to_numpy(),
        "linear_prior": linear_prior,
        "windowed_match": windowed_match,
        "match_minus_prior": windowed_match - linear_prior,
        "dist_from_known_boundary": eval_rows["MD"].to_numpy() - known_md_max,
        "eval_zone_frac": (np.arange(n_eval) + 1) / n_eval,
        "known_zone_rows": len(known),
    })

    if has_target and "TVT" in hw.columns:
        df["target"] = eval_rows["TVT"].to_numpy()

    return df


def build_dataset(split_dir, wells, has_target):
    frames, failed = [], []
    for well in wells:
        try:
            f = build_well_features(well, split_dir, has_target)
            if f is not None:
                frames.append(f)
        except Exception as exc:  # noqa: BLE001
            print(f"  well {well} failed ({exc}); skipping")
            failed.append(well)
    if not frames:
        return pd.DataFrame(), failed
    return pd.concat(frames, ignore_index=True), failed

In [ ]:
class FormationPlaneKNN:
    """Locally-weighted plane fit of each formation's depth vs (X, Y), using
    the K nearest OTHER wells (inverse-distance weighted). Adapted from
    lightningv08/rogii-dual-pipeline-self-verifying."""

    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            path = os.path.join(data_dir, f"{wid}__horizontal_well.csv")
            try:
                df = pd.read_csv(path, usecols=["X", "Y"] + FORMATIONS).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            row = {"wid": wid, "x": float(df.X.median()), "y": float(df.Y.median())}
            for c in FORMATIONS:
                row[f"{c}_m"] = float(df[c].median())
            rows.append(row)

        self.df = pd.DataFrame(rows)
        self.wmap = {w: i for i, w in enumerate(self.df.wid)}
        xy = self.df[["x", "y"]].to_numpy()
        self.scale = np.where(xy.std(0) < 1e-3, 1.0, xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df.x.to_numpy()
        self.ya = self.df.y.to_numpy()
        self.fa = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale
        nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap:
            dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ordr = np.argpartition(dist, min(k - 1, nf - 1), axis=1)[:, :k]
        dk = np.take_along_axis(dist, ordr, 1)
        ik = np.take_along_axis(idx, ordr, 1)
        vk = np.isfinite(dk)
        w = np.where(vk, 1.0 / (dk + 1e-3), 0.0).astype(np.float64)

        xn, yn, fn = self.xa[ik], self.ya[ik], self.fa[ik]
        wx, wy = w * xn, w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1];       A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2];       A[:, 2, 1] = A[:, 1, 2];       A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-6; A[:, 1, 1] += 1e-6; A[:, 2, 2] += 1e-6

        rhs = np.stack([
            (wx[:, :, None] * fn).sum(1),
            (wy[:, :, None] * fn).sum(1),
            (w[:, :, None] * fn).sum(1),
        ], axis=1)

        coef = np.zeros((len(q), 3, len(FORMATIONS)))
        try:
            coef = np.linalg.solve(A, rhs)
        except np.linalg.LinAlgError:
            for r in range(len(q)):
                try:
                    coef[r] = np.linalg.solve(A[r], rhs[r])
                except np.linalg.LinAlgError:
                    coef[r] = 0.0

        pred = coef[:, 0, :] * q[:, [0]] + coef[:, 1, :] * q[:, [1]] + coef[:, 2, :]
        return pred


def add_spatial_prior_feature(dataset_df, hw_cache, knn_index, guard_ft=SPATIAL_GUARD_FT):
    """Guarded: falls back to linear_prior for any row whose spatial estimate
    disagrees with it by more than guard_ft - a raw, unguarded attempt hit
    catastrophic extrapolation blowups from ill-conditioned local plane fits."""
    out = np.full(len(dataset_df), np.nan)
    for well, group in dataset_df.groupby("well", sort=False):
        hw = hw_cache[well]
        known = hw[hw["TVT_input"].notna()]
        if len(known) < 5:
            continue
        known_xy = known[["X", "Y"]].to_numpy(dtype=float)
        known_form = knn_index.impute(known_xy, self_wid=well)
        known_z = known["Z"].to_numpy(dtype=float)
        known_tvt = known["TVT_input"].to_numpy(dtype=float)
        offsets = np.median(known_tvt[:, None] - (-known_z[:, None] + known_form), axis=0)

        eval_xy = group[["X", "Y"]].to_numpy(dtype=float)
        eval_form = knn_index.impute(eval_xy, self_wid=well)
        eval_z = group["Z"].to_numpy(dtype=float)
        candidate_tvt = -eval_z[:, None] + eval_form + offsets[None, :]
        spatial_prior = np.median(candidate_tvt, axis=1)

        linear_prior = group["linear_prior"].to_numpy(dtype=float)
        unstable = np.abs(spatial_prior - linear_prior) > guard_ft
        spatial_prior = np.where(unstable, linear_prior, spatial_prior)
        out[group.index.to_numpy()] = spatial_prior
    return out

In [ ]:
t0 = time.time()
train_wells = list_wells(TRAIN_DIR)
print(f"Building TRAIN features for {len(train_wells)} wells...")
train_data, train_failed = build_dataset(TRAIN_DIR, train_wells, has_target=True)
train_data = train_data.dropna(subset=["target"]).reset_index(drop=True)
print(f"Train dataset: {train_data.shape}, {len(train_failed)} wells failed, "
      f"built in {time.time()-t0:.1f}s")

print("Building FormationPlaneKNN index from all train wells...")
knn_index = FormationPlaneKNN(train_wells, TRAIN_DIR)
print(f"Index built on {len(knn_index.df)} wells with valid formation columns")

print("Caching per-well horizontal_well.csv for spatial feature building (train)...")
train_hw_cache = {w: pd.read_csv(os.path.join(TRAIN_DIR, f"{w}__horizontal_well.csv")).reset_index(drop=True)
                   for w in train_wells}

print("Computing spatial_tvt_prior for train (leak-free: self-well excluded per query)...")
train_data["spatial_tvt_prior"] = add_spatial_prior_feature(train_data, train_hw_cache, knn_index)
print(f"NaN spatial_tvt_prior: {train_data['spatial_tvt_prior'].isna().sum()} / {len(train_data)}")

GLOBAL_FALLBACK = float(train_data["target"].mean())
train_data["spatial_tvt_prior"] = train_data["spatial_tvt_prior"].fillna(GLOBAL_FALLBACK)

X_train = train_data[AUG_FEATURE_COLS]
y_train = train_data["target"].to_numpy()

model = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
model.fit(X_train, y_train)
print("Model trained on", len(train_data), "rows.")
print(f"GLOBAL_FALLBACK = {GLOBAL_FALLBACK:.2f}")

In [ ]:
test_wells = list_wells(TEST_DIR)
print(f"Building TEST features for {len(test_wells)} wells...")
test_data, test_failed = build_dataset(TEST_DIR, test_wells, has_target=False)
print(f"Test dataset: {test_data.shape}, {len(test_failed)} wells failed")

rows = []
if len(test_data) > 0:
    test_data = test_data.reset_index(drop=True)
    test_hw_cache = {w: pd.read_csv(os.path.join(TEST_DIR, f"{w}__horizontal_well.csv")).reset_index(drop=True)
                      for w in test_wells}
    test_data["spatial_tvt_prior"] = add_spatial_prior_feature(test_data, test_hw_cache, knn_index)
    test_data["spatial_tvt_prior"] = test_data["spatial_tvt_prior"].fillna(GLOBAL_FALLBACK)

    X_test = test_data[AUG_FEATURE_COLS]
    try:
        preds = model.predict(X_test)
    except Exception as exc:  # noqa: BLE001
        print(f"batch predict failed ({exc}); falling back to GLOBAL_FALLBACK for all test rows")
        preds = np.full(len(test_data), GLOBAL_FALLBACK)

    for well, row_idx, pred in zip(test_data["well"], test_data["row_idx"], preds):
        rows.append({"id": f"{well}_{row_idx}", "tvt": pred})

submission = pd.DataFrame(rows, columns=["id", "tvt"])
print(f"built {len(submission)} predictions across "
      f"{len(test_wells) - len(test_failed)} wells ({len(test_failed)} wells failed)")
submission.head()

In [ ]:
sample = pd.read_csv(SAMPLE_SUBMISSION)
submission = submission.drop_duplicates(subset="id", keep="first")

merged = sample[["id"]].merge(submission, on="id", how="left")
n_missing = merged["tvt"].isna().sum()
if n_missing > 0:
    print(f"WARNING: {n_missing} required ids had no prediction - filling with "
          f"GLOBAL_FALLBACK ({GLOBAL_FALLBACK:.2f})")
    merged["tvt"] = merged["tvt"].fillna(GLOBAL_FALLBACK)

extra_ids = set(submission["id"]) - set(sample["id"])
if extra_ids:
    print(f"NOTE: {len(extra_ids)} predicted ids aren't in sample_submission - dropped")

assert len(merged) == len(sample), f"row count still off: {len(merged)} vs {len(sample)}"
assert merged["tvt"].notna().all(), "still have NaNs after fallback fill - should be impossible"

submission = merged
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)
print(f"Total runtime: {time.time()-t0:.1f}s")